In [ ]:
import os
import string
import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
from keras import regularizers

In [ ]:
char_list = string.digits + 'OTDB?'  # 0-9, A, D, O, T, space
char_to_num = {char: i+3 for i, char in enumerate(char_list)}  # Start from 3

# Special tokens
START_TOKEN = 0
END_TOKEN = 1
PADDING_TOKEN = 2

# Add special tokens to conversion
num_to_char = {i+3: char for char, i in zip(char_list, range(len(char_list)))}
num_to_char[START_TOKEN] = '<START>'
num_to_char[END_TOKEN] = '<END>'
num_to_char[PADDING_TOKEN] = '<PAD>'

# Vocabulary size includes all characters + special tokens
num_classes = len(char_list) + 3  # chars + START + END + PAD

# Image and sequence parameters
TARGET_HEIGHT = 64
TARGET_WIDTH = 256
MAX_LABEL_LEN = 64
print(f"Vocabulary size: {num_classes}")
print(f"Character mapping: {char_to_num}")
print(f"Image size: {TARGET_HEIGHT}x{TARGET_WIDTH}")

Vocabulary size: 18
Character mapping: {'0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12, 'O': 13, 'T': 14, 'D': 15, 'B': 16, '?': 17}
Image size: 64x256


In [ ]:
# ============= IMAGE PREPROCESSING =============
def prepare_image(image, target_height=TARGET_HEIGHT, target_width=TARGET_WIDTH):
    width, height = image.size

    # Calculate scaling factor to fit within target while preserving aspect ratio
    scale = min(target_width / width, target_height / height)

    if scale < 1.0:  # Only resize if image is larger
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.LANCZOS)
        width, height = new_width, new_height

    # Create a white canvas of target size
    canvas = Image.new('L', (target_width, target_height), 255)

    # Paste the image in the center
    paste_x = (target_width - width) // 2
    paste_y = (target_height - height) // 2
    canvas.paste(image, (paste_x, paste_y))

    return canvas

In [ ]:
# ============= Single IMAGE PREPROCESSING =============

def preprocess_image(image_path, target_height=TARGET_HEIGHT, target_width=TARGET_WIDTH):

    # Load image
    image = Image.open(image_path).convert("L")

    # Resize/pad
    image = prepare_image(image, target_height, target_width)

    # Convert to numpy and normalize
    image_np = np.array(image).astype(np.float32)

    # Normalize to [0, 1]
    image_np = image_np / 255.0

    # Optional: Invert if text is dark on light background
    # image_np = 1.0 - image_np

    # Add channel dimension: (H, W, 1)
    image_np = np.expand_dims(image_np, axis=-1)

    return image_np

In [ ]:
# ============= LABEL ENCODING =============
def encode_label(label_text, max_len=MAX_LABEL_LEN,
                 add_start=True, add_end=True):
    """
    Encode text label to sequence of token IDs.

    Args:
        label_text: Input text string
        max_len: Maximum sequence length (including START/END)
        add_start: Whether to add START token
        add_end: Whether to add END token

    Returns:
         token IDs
    """
    # Clean the text (remove extra spaces, etc.)
    label_text = label_text.strip()

    # Encode characters
    encoded = []

    if add_start:
        encoded.append(START_TOKEN)

    for char in label_text:
        if char in char_to_num:
            encoded.append(char_to_num[char])
        else:
            print(f"Warning: Character '{char}' not in vocabulary, skipping")

    if add_end:
        encoded.append(END_TOKEN)

    # Truncate if too long
    if len(encoded) > max_len:
        print(f"Warning: Label too long ({len(encoded)} > {max_len}), truncating")
        encoded = encoded[:max_len-1] + [END_TOKEN]

    return encoded

In [ ]:
def decode_label(token_ids, remove_special=True):
    decoded = []

    for token_id in token_ids:
        if remove_special and token_id in [START_TOKEN, END_TOKEN, PADDING_TOKEN]:
            continue

        if token_id in num_to_char:
            decoded.append(num_to_char[token_id])

    return ''.join(decoded)

In [ ]:
# ============= DATASET PREPARATION =============
def prepare_data(data_dir, img_height=TARGET_HEIGHT, img_width=TARGET_WIDTH,
                 max_label_len=MAX_LABEL_LEN, verbose=True):
    image_files = [f for f in os.listdir(data_dir) if f.lower().endswith(".jpg")]

    if len(image_files) == 0:
        raise ValueError(f"No .jpg files found in {data_dir}")

    images = []
    labels = []
    skipped = 0

    for img_file in image_files:
        img_path = os.path.join(data_dir, img_file)
        label_path = os.path.splitext(img_path)[0] + ".txt"

        # Check if label file exists
        if not os.path.exists(label_path):
            print(f"Warning: Label file not found for {img_file}, skipping")
            skipped += 1
            continue

        try:
            # Load and preprocess image
            image_np = preprocess_image(img_path, img_height, img_width)

            # Load label
            with open(label_path, "r") as f:
                label_text = f.readline().strip()

            # Skip empty labels
            if not label_text:
                print(f"Warning: Empty label for {img_file}, skipping")
                skipped += 1
                continue

            # Encode label (with START and END tokens)
            label_seq = encode_label(label_text, max_len=max_label_len)

            if verbose:
                print(f"Loaded: {img_file}")
                print(f"  Label text: '{label_text}'")
                print(f"  Label tokens: {label_seq}")
                print(f"  Decoded: '{decode_label(label_seq)}'")

            images.append(image_np)
            labels.append(label_seq)

        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
            skipped += 1
            continue

    if len(images) == 0:
        raise ValueError("No valid images loaded!")

    # Pad labels to max length
    padded_labels = tf.keras.preprocessing.sequence.pad_sequences(
        labels,
        maxlen=max_label_len,
        padding='post',
        value=PADDING_TOKEN
    )

    print(f"\n{'='*60}")
    print(f"Dataset prepared:")
    print(f"  Total files: {len(image_files)}")
    print(f"  Successfully loaded: {len(images)}")
    print(f"  Skipped: {skipped}")
    print(f"  Image shape: {images[0].shape}")
    print(f"  Label shape: {padded_labels.shape}")
    print(f"  Label range: [{padded_labels.min()}, {padded_labels.max()}]")
    print(f"{'='*60}\n")

    return np.array(images), np.array(padded_labels)

In [ ]:
# ============= DATA VALIDATION =============
def validate_dataset(images, labels):

    print("Validating dataset...")

    # Check shapes
    assert len(images) == len(labels), "Mismatch between images and labels"
    assert images.ndim == 4, f"Images should be 4D, got {images.ndim}D"
    assert labels.ndim == 2, f"Labels should be 2D, got {labels.ndim}D"

    # Check value ranges
    assert images.min() >= 0 and images.max() <= 1, \
        f"Images should be normalized to [0,1], got [{images.min()}, {images.max()}]"

    assert labels.min() >= 0 and labels.max() < num_classes, \
        f"Labels should be in [0, {num_classes}), got [{labels.min()}, {labels.max()}]"

    # Check for valid sequences
    for i, label in enumerate(labels[:10]):  # Check first 10
        # Remove padding
        label_no_pad = label[label != PADDING_TOKEN]

        # Check if has START and END
        if len(label_no_pad) > 0:
            if label_no_pad[0] != START_TOKEN:
                print(f"Warning: Label {i} doesn't start with START_TOKEN")

            # Find first END token
            end_indices = np.where(label_no_pad == END_TOKEN)[0]
            if len(end_indices) == 0:
                print(f"Warning: Label {i} doesn't have END_TOKEN")

    # Print statistics
    label_lengths = np.sum(labels != PADDING_TOKEN, axis=1)
    print(f"\nLabel length statistics:")
    print(f"  Min: {label_lengths.min()}")
    print(f"  Max: {label_lengths.max()}")
    print(f"  Mean: {label_lengths.mean():.2f}")
    print(f"  Median: {np.median(label_lengths):.2f}")

    print("\nSample labels (first 5):")
    for i in range(min(5, len(labels))):
        label_no_pad = labels[i][labels[i] != PADDING_TOKEN]
        decoded = decode_label(label_no_pad)
        print(f"  {i}: {label_no_pad} -> '{decoded}'")

    print("\n✓ Dataset validation passed!")

In [ ]:
# ============= USAGE EXAMPLE =============
if __name__ == "__main__":
    # Load data
    data_dir = "/home/green-fin/pythonfiles/Prediction/train/MICR"

    images, labels = prepare_data(data_dir, verbose=True)

    # Validate
    validate_dataset(images, labels)

    # Save configuration for later use
    config = {
        'num_classes': num_classes,
        'padding_token': PADDING_TOKEN,
        # 'start_token': START_TOKEN,
        # 'end_token': END_TOKEN,
        'max_label_len': MAX_LABEL_LEN,
        'img_height': TARGET_HEIGHT,
        'img_width': TARGET_WIDTH,
        'char_to_num': char_to_num,
        'num_to_char': num_to_char,
    }

    print("\nConfiguration for training:")
    print(f"  num_classes = {num_classes}")
    print(f"  padding_token = {PADDING_TOKEN}")
    print(f"  start_token = {START_TOKEN}")
    print(f"  end_token = {END_TOKEN}")
    print(f"  max_length = {MAX_LABEL_LEN}")

    # Ready for training!
    print("\n✓ Data ready for training!")
    print(f"Use: train_model(images, labels, encoder, decoder, ...)")

Loaded: cheque_2182.jpg
  Label text: 'O002210O  T261071315T 7671274O'
  Label tokens: [0, 13, 3, 3, 5, 5, 4, 3, 13, 14, 5, 9, 4, 3, 10, 4, 6, 4, 8, 14, 10, 9, 10, 4, 5, 10, 7, 13, 1]
  Decoded: 'O002210OT261071315T7671274O'
Loaded: cheque_259.jpg
  Label text: 'O0000001023O T061000227T 8669914056O'
  Label tokens: [0, 13, 3, 3, 3, 3, 3, 3, 4, 3, 5, 6, 13, 14, 3, 9, 4, 3, 3, 3, 5, 5, 10, 14, 11, 9, 9, 12, 12, 4, 7, 3, 8, 9, 13, 1]
  Decoded: 'O0000001023OT061000227T8669914056O'
Loaded: cheque_3942.jpg
  Label text: 'O0010001010O T084201294T 4010404403O'
  Label tokens: [0, 13, 3, 3, 4, 3, 3, 3, 4, 3, 4, 3, 13, 14, 3, 11, 7, 5, 3, 4, 5, 12, 7, 14, 7, 3, 4, 3, 7, 3, 7, 7, 3, 6, 13, 1]
  Decoded: 'O0010001010OT084201294T4010404403O'
Loaded: cheque_2780.jpg
  Label text: 'T061000052T 334005022248O6658'
  Label tokens: [0, 14, 3, 9, 4, 3, 3, 3, 3, 8, 5, 14, 6, 6, 7, 3, 3, 8, 3, 5, 5, 5, 7, 11, 13, 9, 9, 8, 11, 1]
  Decoded: 'T061000052T334005022248O6658'
Loaded: cheque_1665.jpg
  Label text

In [ ]:
#     config = {
#         'num_classes': num_classes,
#         'padding_token': PADDING_TOKEN,
#         'start_token': START_TOKEN,
#         'end_token': END_TOKEN,
#         'max_label_len': MAX_LABEL_LEN,
#         'img_height': TARGET_HEIGHT,
#         'img_width': TARGET_WIDTH,
#         'char_to_num': char_to_num,
#         'num_to_char': num_to_char,
#     }
# CONFIG = {
#     'batch_size': 8,
#     'epochs': 50,
#     'initial_lr': 0.001,
#     'embedding_dim': 256,
#     'decoder_units': 512,
#     'attention_units': 256,
#     'dropout_rate': 0.0,
#     'gradient_clip': 1.0,
#     'teacher_forcing_decay': 0.02,
# }

In [ ]:
# ============= CONFIGURATION =============
CONFIG = {
    # Model Architecture
    'conv_filters': [64, 128, 256, 512],
    'kernel_size': (3, 3),
    'embedding_dim': 512,
    'decoder_units': 128,
    'attention_units': 256,
    'dropout_rate': 0.3,


    # Image preprocessing
    'img_height': TARGET_HEIGHT,
    'img_width': TARGET_WIDTH,
    'normalize': True,
    # Model
    'batch_size': 2,
    'epochs': 100,
    'initial_lr': 0.001,
    'gradient_clip': 4.0,
    'teacher_forcing_decay': 0.02,
}

In [ ]:
class ImprovedFeatureEncoder(tf.keras.Model):
    def __init__(self, config):
        super(ImprovedFeatureEncoder, self).__init__()
        filters = config['conv_filters']
        kernel = config['kernel_size']
        dropout = config['dropout_rate']

        # Block 1: 64 filters (cleaned up unused layers)
        self.conv1_1 = layers.Conv2D(filters[0], kernel, padding='same', kernel_initializer='he_normal')
        self.bn1_1 = layers.BatchNormalization()
        self.conv1_2 = layers.Conv2D(filters[0], kernel, padding='same', kernel_initializer='he_normal')
        self.bn1_2 = layers.BatchNormalization()
        self.pool1 = layers.MaxPooling2D((2, 2))
        self.dropout1 = layers.SpatialDropout2D(dropout)

        # Block 2: 128 filters
        self.conv2_1 = layers.Conv2D(filters[1], kernel, padding='same', kernel_initializer='he_normal')
        self.bn2_1 = layers.BatchNormalization()
        self.conv2_2 = layers.Conv2D(filters[1], kernel, padding='same', kernel_initializer='he_normal')
        self.bn2_2 = layers.BatchNormalization()
        self.pool2 = layers.MaxPooling2D((2, 2))
        self.dropout2 = layers.SpatialDropout2D(dropout)

        # Block 3: 256 filters
        self.conv3_1 = layers.Conv2D(filters[2], kernel, padding='same', kernel_initializer='he_normal')
        self.bn3_1 = layers.BatchNormalization()
        self.conv3_2 = layers.Conv2D(filters[2], kernel, padding='same', kernel_initializer='he_normal')
        self.bn3_2 = layers.BatchNormalization()
        self.pool3 = layers.MaxPooling2D((1, 2))
        self.dropout3 = layers.SpatialDropout2D(dropout)

        # Block 4: 512 filters
        self.conv4_1 = layers.Conv2D(filters[3], kernel, padding='same', kernel_initializer='he_normal')
        self.bn4_1 = layers.BatchNormalization()
        self.conv4_2 = layers.Conv2D(filters[3], kernel, padding='same', kernel_initializer='he_normal')
        # self.conv4_2 = layers.Conv2D(filters[3], kernel, padding='same', kernel_initializer='he_normal')
        self.bn4_2 = layers.BatchNormalization()
        self.pool4 = layers.MaxPooling2D((2, 1))  # Only pool height
        self.dropout4 = layers.SpatialDropout2D(dropout)

        # Optional Block 5 (uncomment if needed, but was commented in original)
        self.conv5_1 = layers.Conv2D(filters[3], kernel, padding='same', kernel_initializer='he_normal')
        self.bn5_1 = layers.BatchNormalization()
        self.conv5_2 = layers.Conv2D(filters[3], kernel, padding='same', kernel_initializer='he_normal')
        self.bn5_2 = layers.BatchNormalization()
        self.pool5 = layers.MaxPooling2D((1, 1))  # No pooling
        self.dropout5 = layers.SpatialDropout2D(dropout)

        self.permute = layers.Permute((2, 1, 3))  # For sequence: width as time

    def call(self, inputs, training=False):
        # Block 1
        x = self.conv1_1(inputs)
        # x = self.conv1_2(x)
        x = layers.Activation('relu')(x)
        x = self.bn1_2(x, training=training)
        x = self.pool1(x)

        # Block 2
        x = self.conv2_1(x)
        # x = self.conv2_2(x)
        x = layers.Activation('relu')(x)
        x = self.bn2_2(x, training=training)
        x = self.pool2(x)

        # Block 3
        x = self.conv3_1(x)
        # x = self.conv3_2(x)
        x = layers.Activation('relu')(x)
        x = self.bn3_2(x, training=training)

        # Block 4
        x = self.conv4_1(x)
        # x = self.conv4_2(x)
        x = self.bn4_2(x, training=training)
        x = layers.Activation('relu')(x)
        x = self.pool4(x)


        # Reshape for sequence (width as time)
        x = self.permute(x)
        b, t, f, c = tf.shape(x)[0], x.shape[1], x.shape[2], x.shape[3]
        x = tf.reshape(x, [b, t, f * c])
        return x

In [ ]:
# ============= IMPROVED DECODER =============
class ImprovedAttentionDecoder(tf.keras.Model):
    def __init__(self, vocab_size, config):
        super(ImprovedAttentionDecoder, self).__init__()
        self.units = config['decoder_units']
        dropout = config['dropout_rate']

        self.embedding = layers.Embedding(vocab_size, config['embedding_dim'], mask_zero=True)
        self.dropout_emb = layers.Dropout(dropout)

        # self.lstm = tf.keras.layers.Bidirectional(layers.LSTM(units=self.units, return_sequences=True))s
        # self.lstm1 = layers.LSTM(units=self.units, return_sequences=True)

        # Use bidirectional GRU (remove custom activation)
        self.gru = layers.GRU(
            units=self.units,
            return_sequences=True,
            return_state=True,
            recurrent_dropout=dropout
        )

        self.attention = BahdanauAttention(config['attention_units'])

        self.fcc = layers.Dense(self.units, activation='gelu')
        self.fc = layers.Dense(self.units // 2, activation='gelu')
        self.dropout_fc = layers.Dropout(dropout)
        self.fc_out = layers.Dense(vocab_size)
        self.act = layers.Activation('relu')
        # self.reshape = layers.Reshape((-1, 512))  # (time_steps, features)

        # ============= Bidirectional LSTM =============
        self.rnn1 = layers.Bidirectional(layers.LSTM(self.units*2, return_sequences=True, dropout=0.3))
        self.rnn2 = layers.Bidirectional(layers.LSTM(self.units, return_sequences=True, dropout=0.3))


    def call(self, x, features, hidden, training=True):
        # features: (B, T, F)
        context_vector, attention_weights = self.attention(features, hidden)

        # Optional embedding for input tokens (if using teacher forcing)
        # x = self.embedding(x)
        # x = self.dropout_emb(x, training=training)

        # Concatenate if you use token embeddings too
        # x = tf.concat([context_vector, x], axis=-1)

        # Use the full sequence context
        x = self.rnn1(context_vector)
        # x = self.act(x)
        x = self.rnn2(x)
        # x = self.dropout_fc(x, training=training)

        x = self.fc_out(x)  # (B, T, vocab_size)

        state = None  # can be last hidden if you need it
        return x, state, attention_weights
    def initialize_hidden_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

In [ ]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, features, hidden):
        # features: (B, T, F)
        # hidden:   (B, units)
        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))
        attention_weights = tf.nn.softmax(self.V(score), axis=1)  # (B, T, 1)

        # Context for each time step (still (B, T, F))
        context_vector = attention_weights * features
        return context_vector, attention_weights

In [ ]:
# # Learning rate schedule
# lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
#     initial_learning_rate=CONFIG['initial_lr'],
#     decay_steps=500,
#     decay_rate=0.91,
#     # staircase=True
# )

# # Or use callback
# reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
#     monitor='loss',
#     factor=0.4,
#     patience=3,
#     # min_lr=1e-6,
#     verbose=1
# )

In [ ]:
# Initialize models
encoder = ImprovedFeatureEncoder(CONFIG)
decoder = ImprovedAttentionDecoder(
    vocab_size=num_classes,  # Your vocab size
    config=CONFIG
)

# optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

In [ ]:
decoder.summary()

Model: "improved_attention_decoder_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bahdanau_attention_5            │ ?                      │   0 (unbuilt) │
│ (BahdanauAttention)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5257 (Activation)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_11                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def masked_accuracy(real, pred):
    # real: [batch, seq_len], pred: [batch, seq_len, vocab_size]
    mask = tf.cast(tf.math.not_equal(real, 0), tf.float32)
    pred_ids = tf.argmax(pred, axis=-1, output_type=real.dtype)  # shape: [batch, seq_len]
    correct = tf.cast(tf.equal(real, pred_ids), tf.float32)
    correct *= mask
    return tf.reduce_sum(correct) / tf.reduce_sum(mask)


In [ ]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def masked_loss(y_true, y_pred, padding_token):
    loss = loss_object(y_true=y_true, y_pred=y_pred)
    mask = tf.cast(tf.not_equal(y_true, padding_token), dtype=loss.dtype)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

In [ ]:
# # ============= TRAINING STEP WITH TEACHER FORCING =============
# # @tf.function
# def train_step(tensor, target, encoder, decoder, optimizer, padding_token, teacher_forcing_ratio=0.5):
#     batch_size = tf.shape(tensor)[0]
#     seq_length = tf.shape(target)[1]

#     with tf.GradientTape() as tape:
#         # Encode
#         features = encoder(tensor, training=False)

#         # Initialize decoder
#         hidden = decoder.initialize_hidden_state(batch_size)
#         # dec_input = tf.expand_dims([0] * batch_size, 1)  # Start token
#         # dec_input = tf.expand_dims([0] * batch_size, 1)
#         dec_input = tf.expand_dims(tf.fill([batch_size], START_TOKEN), 1)

#         loss = 0.0

#         # predictions, hidden, _ = decoder(dec_input, features, hidden, training=True)
#         # loss_ = loss_fn(target, predictions)
#         # mask = tf.cast(tf.not_equal(target, padding_token), dtype=loss_.dtype)
#         # loss = tf.reduce_sum(loss_ * mask) / tf.reduce_sum(mask)
#         # loss += tf.reduce_sum(loss_t * mask)


#         # Decode with teacher forcing
#         for t in range(seq_length):
#             predictions, hidden, _ = decoder(dec_input, features, hidden, training=True)
#             # print(predictions.shape)


#             # Calculate loss for this timestep
#             print("Target:",target[:, t],"Prediction:",np.argmax(predictions[:, 0, :]))
#             loss_t = loss_fn(target[:, t], predictions[:, 0, :])
#             mask = tf.cast(tf.not_equal(target[:, t], padding_token), dtype=loss_t.dtype)
#             loss += tf.reduce_sum(loss_t * mask)

#             # Teacher forcing: use ground truth or prediction
#             # dec_input = tf.expand_dims(target[:, t], 1)
#             use_teacher_forcing = tf.random.uniform([]) < teacher_forcing_ratio
#             dec_input = tf.cond(use_teacher_forcing, lambda:tf.expand_dims(target[:,t], 1), lambda: tf.expand_dims(tf.argmax(predictions[:, 0, :], axis=-1, output_type=tf.int32), 1))
#             # if use_teacher_forcing:
#             #     dec_input = tf.expand_dims(target[:, t], 1)
#             # else:
#             #     predicted_id = tf.argmax(predictions[:, 0, :], axis=-1, output_type=tf.int32)
#             #     dec_input = tf.expand_dims(predicted_id, 1)

#         # Average loss
#         total_mask = tf.cast(tf.not_equal(target, padding_token), dtype=tf.float32)
#         loss = loss / tf.reduce_sum(total_mask)

#         # lossreg = total_loss / (total_mask + 1e-8)

#         # # Add regularization loss
#         # reg_loss = tf.add_n([tf.nn.l2_loss(v) for v in encoder.trainable_variables
#         #                      if 'kernel' in v.name]) * 1e-5

#     # Compute and apply gradients with clipping
#     trainable_vars = encoder.trainable_variables + decoder.trainable_variables
#     gradients = tape.gradient(loss, trainable_vars)
#     # gradients, _ = tf.clip_by_global_norm(gradients, CONFIG['gradient_clip'])
#     optimizer.apply_gradients(zip(gradients, trainable_vars))

#     return loss

In [ ]:
@tf.function
def train_step(tensor, target, encoder, decoder, optimizer, padding_token, teacher_forcing_ratio=0.5):
    batch_size = tf.shape(tensor)[0]

    with tf.GradientTape() as tape:
        features = encoder(tensor, training=False)
        hidden = decoder.initialize_hidden_state(batch_size)
        dec_input = tf.expand_dims(tf.fill([batch_size], START_TOKEN), 1)

        predictions, hidden, _ = decoder(dec_input, features, hidden, training=True)
        loss = masked_loss(target, predictions, padding_token)
        # loss_ = loss_fn(target, predictions)
        # mask = tf.cast(tf.not_equal(target, padding_token), dtype=loss_.dtype)
        # loss = tf.reduce_sum(loss_ * mask) / tf.reduce_sum(mask)

    trainable_vars = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)
    gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
    optimizer.apply_gradients(zip(gradients, trainable_vars))
    acc = masked_accuracy(target, predictions)
    print("Accuracy:",acc)
    return loss


In [ ]:
def inference(image, encoder, decoder, max_length, start_token=START_TOKEN, end_token=END_TOKEN, padding_token=PADDING_TOKEN):
    image = tf.expand_dims(image, 0) if len(image.shape) == 3 else tf.expand_dims(image, 0)  # Add batch dim
    image = tf.convert_to_tensor(image, dtype=tf.float32)

    features = encoder(image, training=False)
    batch_size = tf.shape(image)[0]

    hidden = decoder.initialize_hidden_state(batch_size)

    # Prepare decoder input (could be start tokens or a fixed tensor as your decoder needs)
    dec_input = tf.expand_dims(tf.fill([batch_size], start_token), 1)

    # Predict the whole sequence at once
    predictions, hidden, attention_weights = decoder(dec_input, features, hidden, training=False)
    # predictions shape: (batch_size, seq_len, vocab_size)

    predicted_ids = tf.argmax(predictions, axis=-1).numpy()  # (batch_size, seq_len)

    return predicted_ids[0], attention_weights.numpy()  # return first batch sample


In [ ]:
# # ============= INFERENCE FUNCTION =============
# def inference(image, encoder, decoder, max_length, start_token=START_TOKEN, end_token=END_TOKEN, padding_token=PADDING_TOKEN):
#     """
#     Arguments Required:
#         image: Input image tensor (H, W, 1) or (H, W)
#         encoder: Trained encoder model
#         decoder: Trained decoder model
#         max_length: Maximum sequence length
#         start_token: Token ID for sequence start
#         end_token: Token ID for sequence end
#         padding_token: Token ID for padding

#     Returns Arguments:
#         predicted_sequence: List of predicted token IDs
#         attention_plot: Attention weights for visualization
#     """
#     # Add batch dimension if needed
#     if len(image.shape) == 2:
#         image = image[..., np.newaxis]
#     if len(image.shape) == 3:
#         image = np.expand_dims(image, 0)

#     image = tf.convert_to_tensor(image, dtype=tf.float32)

#     # Encode
#     features = encoder(image, training=False)

#     # Initialize decoder
#     hidden = decoder.initialize_hidden_state(1)
#     dec_input = tf.expand_dims([start_token], 1)

#     result = []
#     attention_weights_list = []

#     for t in range(max_length):
#         predictions, hidden, attention_weights = decoder(dec_input, features, hidden, training=False)
#         attention_weights_list.append(attention_weights.numpy())

#         # Get predicted token
#         predicted_id = tf.argmax(predictions[0, 0, :]).numpy()
#         result.append(predicted_id)

#         # Stop if end token is predicted
#         if predicted_id == end_token:
#             break

#         # Use prediction as next input
#         dec_input = tf.expand_dims([predicted_id], 1)

#     attention_plot = np.concatenate(attention_weights_list, axis=1)
#     return result, attention_plot

In [ ]:
# # ============= ACCURACY CALCULATION =============
# def calculate_accuracy(predictions, labels, padding_token):

#     total_chars = 0
#     correct_chars = 0
#     total_seqs = 0
#     correct_seqs = 0

#     for pred, label in zip(predictions, labels):
#         # Remove padding from label
#         label_no_pad = [l for l in label if l != padding_token]

#         # Compare sequences
#         seq_match = True
#         for i, l in enumerate(label_no_pad):
#             if i < len(pred):
#                 if pred[i] == l:
#                     correct_chars += 1
#                 else:
#                     seq_match = False
#             else:
#                 seq_match = False
#             total_chars += 1

#         # Check if lengths match
#         if len(pred) != len(label_no_pad):
#             seq_match = False

#         if seq_match:
#             correct_seqs += 1
#         total_seqs += 1

#     char_acc = correct_chars / total_chars if total_chars > 0 else 0
#     seq_acc = correct_seqs / total_seqs if total_seqs > 0 else 0

#     return char_acc, seq_acc

In [ ]:
def calculate_accuracy(predictions, labels, padding_token):
    total_chars = 0
    correct_chars = 0
    total_seqs = 0
    correct_seqs = 0

    for pred, label in zip(predictions, labels):
        # Remove padding from label
        label_no_pad = [l for l in label if l != padding_token]

        # Optionally, remove padding from prediction as well
        pred_no_pad = [p for p in pred if p != padding_token]

        # Character accuracy: count matches up to max length of label and pred
        max_len = max(len(label_no_pad), len(pred_no_pad))
        for i in range(max_len):
            l = label_no_pad[i] if i < len(label_no_pad) else None
            p = pred_no_pad[i] if i < len(pred_no_pad) else None
            if l is not None:
                total_chars += 1
                if p == l:
                    correct_chars += 1

        # Sequence accuracy: exact match including length and content (excluding padding)
        seq_match = (label_no_pad == pred_no_pad)
        if seq_match:
            correct_seqs += 1
        total_seqs += 1

    char_acc = correct_chars / total_chars if total_chars > 0 else 0
    seq_acc = correct_seqs / total_seqs if total_seqs > 0 else 0

    return char_acc, seq_acc


In [ ]:
# ============= VALIDATION FUNCTION =============
def validate(images, labels, encoder, decoder, max_length, padding_token, num_samples=None):

    if num_samples is None:
        num_samples = len(images)

    predictions = []

    for i in range(min(num_samples, len(images))):
        img = images[i]
        pred, _ = inference(img, encoder, decoder, max_length,
                           start_token=0, end_token=1, padding_token=padding_token)
        predictions.append(pred)

    char_acc, seq_acc = calculate_accuracy(predictions[:num_samples],
                                           labels[:num_samples],
                                           padding_token)

    return char_acc, seq_acc, predictions

In [ ]:
# ============= MAIN TRAINING LOOP =============
def train_model(images, labels, encoder, decoder, optimizer,
                epochs, batch_size, padding_token, max_length,
                val_images=None, val_labels=None):

    best_val_acc = 0.0
    teacher_forcing_ratio = 0.8

    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Teacher Forcing Ratio: {teacher_forcing_ratio:.3f}")
        print(f"{'='*60}")

        # Shuffle data
        num_samples = len(images)
        indices = np.arange(num_samples)
        np.random.shuffle(indices)
        images_shuffled = images[indices]
        labels_shuffled = labels[indices]

        # Training
        epoch_loss = 0
        num_batches = 0

        for i in range(0, num_samples, batch_size):
            batch_images = images_shuffled[i:i+batch_size]
            batch_labels = labels_shuffled[i:i+batch_size]

            # Add channel dimension if needed
            if len(batch_images.shape) == 3:
                batch_images = batch_images[..., np.newaxis]

            # Convert to tensors
            batch_images = tf.convert_to_tensor(batch_images, dtype=tf.float32)
            batch_labels = tf.convert_to_tensor(batch_labels, dtype=tf.int32)

            # Train step
            loss = train_step(batch_images, batch_labels, encoder, decoder,
                            optimizer, padding_token, teacher_forcing_ratio)

            epoch_loss += loss.numpy()
            num_batches += 1

            # Progress
            if (num_batches % 10) == 0:
                # print(f"Batch {num_batches}/{num_samples//batch_size} | "
                #       f"Loss: {loss.numpy():.4f}", end='\r')
                print(f"\033[92mBatch {num_batches}/{num_samples//batch_size} | Loss: {loss.numpy():.4f}\033[0m", end='\r')


        avg_loss = epoch_loss / num_batches
        print(f"\nTraining Loss: {avg_loss:.4f}")
        mlflow.log_metric("train_loss", float(avg_loss), step=epoch)
    # mlflow.log_metric("train_accuracy", float(acc), step=epoch)

        # Validation
        if val_images is not None and val_labels is not None:
            print("\nValidating...")
            char_acc, seq_acc, sample_preds = validate(
                val_images, val_labels, encoder, decoder,
                max_length, padding_token, num_samples=min(100, len(val_images))
            )

            # print(f"Validation - Char Accuracy: {char_acc*100:.2f}% | "
            #       f"Seq Accuracy: {seq_acc*100:.2f}%")
            print(f"Validation - Char Accuracy: \033[92m{char_acc*100:.2f}%\033[0m | "
                  f"Seq Accuracy: \033[92m{seq_acc*100:.2f}%\033[0m")


            # Show sample predictions
            print("\nSample Predictions:")
            for j in range(min(3, len(sample_preds))):
                pred = sample_preds[j]
                label = [l for l in val_labels[j] if l != padding_token]
                pred = [l for l in sample_preds[j] if l != padding_token]
                print(f"  True: {label}")
                print(f"  Pred: {pred}")
                print("="*10)

            # Save best model
            if seq_acc > best_val_acc:
                best_val_acc = seq_acc
                print(f"New best validation accuracy! Saving model...")
                encoder.save_weights('best_encoder.h5')
                decoder.save_weights('best_decoder.h5')

        # Decay teacher forcing
        if epoch < 10:
            teacher_forcing_ratio = 0.5
        else:
            teacher_forcing_ratio = max(0.1, teacher_forcing_ratio - CONFIG['teacher_forcing_decay'])
        # teacher_forcing_ratio = max(0.2, teacher_forcing_ratio - CONFIG['teacher_forcing_decay'])

        print(f"{'='*60}")

In [ ]:
if __name__ == "__main__":

    import mlflow
    import mlflow.tensorflow

    mlflow.set_experiment("Image_Captioning_Seq2Seq")

    with mlflow.start_run(run_name="custom_tf_training"):
        # Log parameters
        mlflow.log_params({
            "epochs": CONFIG['epochs'],
            "batch_size": CONFIG['batch_size'],
            "initial_lr": CONFIG['initial_lr'],
        })

        # Optimizer with learning rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=1e-3,
            decay_steps=10000,
            decay_rate=0.90
        )
        # optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        # optimizer = tf.keras.optimizers.AdamW(1e-2, weight_decay=1e-3)
        # optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)#(learning_rate=lr_schedule)
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        # Split data into train/val (80/20)
        split_idx = int(0.8 * len(images))
        train_images, val_images = images[:split_idx], images[split_idx:]
        train_labels, val_labels = labels[:split_idx], labels[split_idx:]

        # Train
        train_model(
            images=train_images,
            labels=train_labels,
            encoder=encoder,
            decoder=decoder,
            optimizer=optimizer,
            epochs=CONFIG['epochs'],
            batch_size=CONFIG['batch_size'],
            padding_token=PADDING_TOKEN,
            max_length=MAX_LABEL_LEN,
            val_images=val_images,
            val_labels=val_labels
        )

        print("\nTraining complete!")

        # ============= TEST INFERENCE =============
        print("\nTesting inference on a sample image...")
        test_img = val_images[0]
        predicted_seq, attention = inference(
            test_img, encoder, decoder, MAX_LABEL_LEN,
            start_token=START_TOKEN, end_token=END_TOKEN, padding_token=PADDING_TOKEN
        )

        true_label = [l for l in val_labels[0] if l != PADDING_TOKEN]
        predicted_seq = [l for l in predicted_seq if l != PADDING_TOKEN]
        print(f"True sequence: {true_label}")
        print(f"Predicted sequence: {predicted_seq}")
        mlflow.tensorflow.log_model(tf.keras.models.Model(), artifact_path="model")


Epoch 1/100
Teacher Forcing Ratio: 0.800


/home/green-fin/anaconda3/lib/python3.12/site-packages/keras/src/layers/layer.py:396: UserWarning: `build()` was called on layer 'improved_feature_encoder_5', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/home/green-fin/anaconda3/lib/python3.12/site-packages/keras/src/layers/layer.py:396: UserWarning: `build()` was called on layer 'improved_attention_decoder_5', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Accuracy: Tensor("truediv_1:0", shape=(), dtype=float32)
Accuracy: Tensor("truediv_1:0", shape=(), dtype=float32)
Batch 1920/1928 | Loss: 1.0724
Training Loss: 1.1828

Validating...
Validation - Char Accuracy: 33.29% | Seq Accuracy: 0.00%

Sample Predictions:
  True: [0, 13, 3, 3, 5, 3, 11, 6, 13, 14, 3, 8, 7, 3, 3, 3, 3, 6, 3, 14, 8, 6, 7, 9, 8, 11, 7, 4, 4, 7, 13, 1]
  Pred: [0, 13, 3, 3, 4, 3, 3, 3, 13, 14, 3, 9, 4, 3, 3, 3, 3, 5, 14, 14, 3, 3, 3, 3, 3, 3, 3, 3]
  True: [0, 14, 3, 9, 4, 3, 3, 3, 3, 8, 5, 14, 6, 6, 7, 3, 6, 12, 11, 12, 8, 6, 3, 8, 13, 4, 11, 5, 5, 1]
  Pred: [0, 13, 3, 3, 4, 3, 3, 3, 13, 14, 3, 9, 4, 3, 3, 3, 3, 5, 14, 14, 3, 3, 3, 3, 3, 3, 3, 3]
  True: [0, 14, 3, 9, 7, 4, 3, 5, 3, 10, 3, 14, 3, 3, 5, 5, 5, 13, 3, 3, 5, 5, 6, 7, 9, 7, 6, 13, 1]
  Pred: [0, 13, 3, 3, 4, 3, 3, 3, 13, 14, 3, 9, 4, 3, 3, 3, 3, 5, 14, 14, 3, 3, 3, 3, 3, 3, 3, 3]

Epoch 2/100
Teacher Forcing Ratio: 0.500
Accuracy: Tensor("truediv_1:0", shape=(), dtype=float32)
Batch 1920/1928 | Loss: 0.91